In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (2).json


{'kaggle (2).json': b'{"username":"pragati2207","key":"adb5455aae9b601b405c70e1c93d9107"}'}

In [2]:
import os

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!pip install kaggle

In [4]:
!kaggle datasets download -d sumanbera19/bengaluru-house-price-dataset

Dataset URL: https://www.kaggle.com/datasets/sumanbera19/bengaluru-house-price-dataset
License(s): CC0-1.0
bengaluru-house-price-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
!unzip bengaluru-house-price-dataset.zip

Archive:  bengaluru-house-price-dataset.zip
replace bengaluru_house_prices.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: bengaluru_house_prices.csv  


In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [7]:
df = pd.read_csv('bengaluru_house_prices.csv')
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [8]:
df.shape
df.columns
df.isnull().sum()

,0
area_type,0
availability,0
location,1
size,16
society,5502
total_sqft,0
bath,73
balcony,609
price,0


In [9]:
df = df.drop(['area_type', 'availability', 'society', 'balcony'], axis=1)

In [10]:
df = df.dropna()

In [11]:
df['bhk'] = df['size'].apply(lambda x: int(x.split(' ')[0]))
df = df.drop('size', axis=1)

In [12]:
def convert_sqft(x):
    if isinstance(x, str) and '-' in x:
        a, b = x.split('-')
        return (float(a) + float(b)) / 2
    try:
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df = df.dropna()

In [13]:
counts = df['location'].value_counts()

def simplify_location(x):
    if counts[x] > 10:
        return x
    else:
        return 'other'

df['location'] = df['location'].apply(simplify_location)

In [14]:
df = pd.get_dummies(df, columns=['location'], drop_first=True)

In [15]:
X = df.drop('price', axis=1)
y = df['price']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [18]:
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

Mean Squared Error: 8659.493275090583
